# Frozen experiment configuration regression tests

Setup is explicit; existing assets are verified before reuse.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import json,tempfile,unittest,copy
from pathlib import Path
from contract import ROOT,resolve
class Checks(unittest.TestCase):
    def test_real_budgets(self):
        for name,budget,episodes in [('checkpoint_check',4096,2),('task_baseline',51200,10),('max_support_baseline',51200,10),('random_baseline',0,10)]:
            result=resolve(ROOT/f'configs/experiments/{name}.yaml')
            self.assertEqual(result['budget_decisions'],budget);self.assertEqual(result['evaluation_episodes'],episodes)
    def test_contract_drift_rejected(self):
        base=json.loads((ROOT/'configs/experiments/task_baseline.yaml').read_text())
        for key,value in [('budget_decisions',1000000),('horizon_decisions',601),('policy_initialization_seed',999),('window_decisions',14)]:
            config=copy.deepcopy(base);config[key]=value
            with tempfile.TemporaryDirectory() as temp:
                path=Path(temp)/'bad.json';path.write_text(json.dumps(config))
                with self.assertRaises(ValueError):resolve(path)
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Frozen configuration cases passed:',result.testsRun)


test_contract_drift_rejected (__main__.Checks.test_contract_drift_rejected) ... 

ok


test_real_budgets (__main__.Checks.test_real_budgets) ... 

Frozen runtime contract definitions/execution completed.


ok


----------------------------------------------------------------------
Ran 2 tests in 0.487s

OK


Frozen configuration cases passed: 2
